Extra Analysis of Evaluate PPO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
!ls -la

/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
total 41
drwx------ 2 root root  4096 Sep 25 15:17 code
drwx------ 2 root root  4096 Sep 25 15:04 .git
-rw------- 1 root root 13586 Oct 31 06:01 github_terminal.ipynb
-rw------- 1 root root    33 Sep 26 19:25 .gitignore
drwx------ 2 root root  4096 Sep 25 15:17 models
drwx------ 2 root root  4096 Oct 31 02:25 OLD
-rw------- 1 root root  2348 Sep 29 02:21 README.md
drwx------ 2 root root  4096 Sep 25 15:17 results
drwx------ 2 root root  4096 Oct 18 05:10 videos


In [ ]:
!pip install gymnasium[atari,accept-rom-license] ale-py sb3_contrib stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 12.9 MB/s eta 0:00:00


In [ ]:
import os
import torch
import gymnasium as gym
import stable_baselines3
import ale_py
import numpy as np
import random

# RL Algorithm
from stable_baselines3 import PPO

# For debugging
from stable_baselines3.common.monitor import Monitor
import time

# Action masking
from gymnasium import ActionWrapper
from stable_baselines3.common.atari_wrappers import AtariWrapper

# Vector environment
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv

# Visualization
import moviepy.editor as mpy
from IPython.display import HTML
from base64 import b64encode

print("All imports working")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294

All imports working


In [ ]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


Visualize Training Curves

Create Environment

In [ ]:
# Create Bowling environment
env = gym.make("ALE/Bowling-v5", render_mode="human")
observation, info = env.reset()

action_space = env.action_space
print(f"Action space: {action_space}")
print("Number of actions:", action_space.n, "\n")

action_meanings = env.unwrapped.get_action_meanings()
print("Action meanings:", action_meanings, "\n")

obs_space = env.observation_space
print(f"Observation shape: {observation.shape}")
print(f"Observation space: {obs_space}")

env.close()
del env
print("Bowling env closed")

Action space: Discrete(6)
Number of actions: 6 

Action meanings: ['NOOP', 'FIRE', 'UP', 'DOWN', 'UPFIRE', 'DOWNFIRE'] 

Observation shape: (210, 160, 3)
Observation space: Box(0, 255, (210, 160, 3), uint8)
Bowling env closed


In [ ]:
class ActionReducer(ActionWrapper):
  def __init__(self, env):
    super().__init__(env)

    # NOOP, FIRE, UP, and DOWN only. No UPFIRE. No DOWNFIRE.
    self.allowed_actions = [0,1,2,3]

    self.action_meanings = ['NOOP', 'FIRE', 'UP', 'DOWN']

    self.action_space = gym.spaces.Discrete(len(self.allowed_actions))


  def action(self, action):
    return self.allowed_actions[action]

    def get_action_meanings(self):
        return self.action_meanings

In [ ]:
game_name = "ALE/Bowling-v5"

In [ ]:
seed = 7
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
def make_env(render_mode=None):
  env = gym.make(game_name, render_mode=render_mode)

  # disable reward clipping
  env = AtariWrapper(env, clip_reward=False)

  env = ActionReducer(env)
  # Monitor should wrap it last. Gives the Mean Episode Length & Reward
  env = Monitor(env)

  return env

In [ ]:
env = DummyVecEnv([lambda: make_env()])
env.seed(seed)
env = VecFrameStack(env, n_stack=4)

Confirm there are only 4 actions

In [ ]:
observation = env.reset()
# info = {}

action_space = env.action_space
print(f"Action space: {action_space}")
print("Number of actions:", action_space.n, "\n")

obs_space = env.observation_space
print(f"Observation shape: {observation.shape}")
print(f"Observation space: {obs_space}")

Action space: Discrete(4)
Number of actions: 4 

Observation shape: (1, 84, 84, 4)
Observation space: Box(0, 255, (84, 84, 4), uint8)


In [ ]:
env.close()
del env

In [ ]:
action_dict = {
  0: "NOOP",
  1: "FIRE",
  2: "UP",
  3: "DOWN",
  4: "UPFIRE",
  5: "DOWNFIRE"
}

Video

In [ ]:
video_env = DummyVecEnv([lambda:make_env("rgb_array")])
video_env.seed(seed)
video_env = VecFrameStack(video_env, n_stack=4)

In [ ]:
model_name = "ppo_10000000"

# Load model with video_env
model = PPO.load(
    f"models/{model_name}",
    env=video_env,
    device="cuda"
)

print("Model loaded")

Wrapping the env in a VecTransposeImage.
Model loaded


In [ ]:
def evaluate_model_and_make_video(model, n_eval_episodes, env):
  """Evaluate model and return mean reward"""

  frames = []
  all_rewards = []
  all_steps = []

  for episode in range(n_eval_episodes):
    obs = env.reset()
    episode_reward = 0
    dones = [False]
    steps = 0

    # Track bowling frames
    bowling_frame = 1
    frame_start_step = 0
    frame_start_score = 0
    last_reward_step = 0

    print(f"EPISODE {episode + 1}\n")

    while not dones[0]:
      action, _ = model.predict(obs, deterministic=True)
      obs, rewards, dones, infos = env.step(action)

      frame = env.envs[0].render()
      frames.append(frame)

      episode_reward += rewards[0]
      steps += 1

      # Only when rewarded
      if rewards[0] > 0:
        # Calculate frame statistics
        steps_in_frame = steps - frame_start_step
        frame_score = rewards[0]
        start_frame = frame_start_step + 1
        end_frame = steps

        print(f"BOWLING FRAME {bowling_frame} COMPLETE")
        print(f"Score this frame: {frame_score}")
        print(f"Total score: {episode_reward}")
        print(f"Steps in frame: {steps_in_frame} (steps {start_frame}-{end_frame})")
        print(f"Last action: {action_dict[action[0]]}\n\n")

        bowling_frame += 1
        frame_start_step = steps
        frame_start_score = episode_reward
        last_reward_step = end_frame

      if dones[0]:
        # CAPTURE ADDITIONAL FRAMES AFTER EPISODE ENDS
        # bc game ends at 179, before it adds 9 to the score to get the final score of 188
        # Capture 50 more frames
        for _ in range(100):
          frame = env.envs[0].render()
          frames.append(frame)

    print(f"EPISODE {episode+1} SUMMARY: Total Reward = {episode_reward:6.1f}, Total Steps = {steps}")

    all_rewards.append(episode_reward)
    all_steps.append(steps)

  return all_rewards, all_steps

In [ ]:
# n_eval_episodes = 3
n_eval_episodes = 1
video_name = f"zzz_PPO_bowling_game_analysis.mp4"
video_save_path = f"/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/{video_name}"

all_rewards, all_steps = evaluate_model_and_make_video(model, n_eval_episodes, video_env, video_save_path)

EPISODE 1

BOWLING FRAME 1 COMPLETE
Score this frame: 20.0
Total score: 20.0
Steps in frame: 78 (steps 1-78)
Last action: FIRE


BOWLING FRAME 2 COMPLETE
Score this frame: 20.0
Total score: 40.0
Steps in frame: 54 (steps 79-132)
Last action: FIRE


BOWLING FRAME 3 COMPLETE
Score this frame: 20.0
Total score: 60.0
Steps in frame: 26 (steps 133-158)
Last action: FIRE


BOWLING FRAME 4 COMPLETE
Score this frame: 20.0
Total score: 80.0
Steps in frame: 54 (steps 159-212)
Last action: FIRE


BOWLING FRAME 5 COMPLETE
Score this frame: 20.0
Total score: 100.0
Steps in frame: 26 (steps 213-238)
Last action: FIRE


BOWLING FRAME 6 COMPLETE
Score this frame: 20.0
Total score: 120.0
Steps in frame: 54 (steps 239-292)
Last action: FIRE


BOWLING FRAME 7 COMPLETE
Score this frame: 20.0
Total score: 140.0
Steps in frame: 26 (steps 293-318)
Last action: FIRE


BOWLING FRAME 8 COMPLETE
Score this frame: 20.0
Total score: 160.0
Steps in frame: 55 (steps 319-373)
Last action: NOOP


BOWLING FRAME 9 COMPL

Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/bowling_game_analysis.mp4


In [ ]:
mean_reward = np.mean(all_rewards)
mean_steps = np.mean(all_steps)
print(f"{mean_reward:.4f} mean reward, {mean_steps:.4f} mean steps")

188.0000 mean reward, 426.0000 mean steps


In [ ]:
# Display the video inline as part of Google Colab
mp4 = open(video_save_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=480 controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
video_env.close()
del video_env

In [ ]:
del model